# CycleDiffusion — Kaggle Layer 4 continuation

This notebook uses the private CycleDiffusion Kaggle Dataset, applies the current research overlay, and runs only the unfinished `speaker_only`, `content_only`, and `hard_gate` controlled ablations. Completed `uniform` and `joint` evidence is reused rather than retrained.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile

input_root = Path('/kaggle/input')
working_root = Path('/kaggle/working')
asset_dirs = [p for p in input_root.rglob('CycleDiffusion_kaggle_assets') if p.is_dir()]
overlay_dirs = [p for p in input_root.rglob('cyclediffusion_kaggle_overlay') if p.is_dir()]
asset_archives = list(input_root.rglob('CycleDiffusion_kaggle_assets.zip'))
overlay_archives = list(input_root.rglob('cyclediffusion_kaggle_overlay.zip'))
assert len(asset_dirs) + len(asset_archives) == 1, (asset_dirs, asset_archives)
assert len(overlay_dirs) + len(overlay_archives) == 1, (overlay_dirs, overlay_archives)
asset_source = asset_dirs[0] if asset_dirs else asset_archives[0]
overlay_source = overlay_dirs[0] if overlay_dirs else overlay_archives[0]
asset_source, overlay_source

In [ ]:
%pip install -q --upgrade 'librosa>=0.10' 'transformers>=4.44' soundfile tgt einops

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before running.'
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
project_root = working_root / 'CycleDiffusion'
if not project_root.exists() and asset_source.is_dir():
    candidates = [p for p in asset_source.rglob('CycleDiffusion') if (p / 'VCTK_2F2M').is_dir()]
    assert len(candidates) == 1, f'Cannot identify extracted project root: {candidates}'
    shutil.copytree(candidates[0], project_root, dirs_exist_ok=True)
elif not project_root.exists():
    with zipfile.ZipFile(asset_source) as zf:
        zf.extractall(working_root)
if not project_root.exists():
    candidates = [p for p in working_root.rglob('VCTK_2F2M') if p.is_dir()]
    assert len(candidates) == 1, f'Cannot identify project root: {candidates}'
    project_root = candidates[0].parent

overlay_root = working_root / '_kaggle_overlay'
if overlay_root.exists():
    shutil.rmtree(overlay_root)
if overlay_source.is_dir():
    shutil.copytree(overlay_source, overlay_root, dirs_exist_ok=True)
else:
    with zipfile.ZipFile(overlay_source) as zf:
        zf.extractall(overlay_root)
for source in (overlay_root / 'project_overlay').iterdir():
    destination = project_root / source.name
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        shutil.copy2(source, destination)

controlled_root = working_root / 'full4_controlled'
formal_metrics = controlled_root / 'formal_evaluation' / 'metrics'
formal_metrics.mkdir(parents=True, exist_ok=True)
shutil.copy2(
    overlay_root / 'formal_evaluation' / 'metrics' / 'formal_eval_raw.csv',
    formal_metrics / 'formal_eval_raw.csv',
)
os.environ['CYCLEDIFFUSION_ROOT'] = str(project_root)
os.environ['CYCLEDIFFUSION_RESULTS_ROOT'] = str(controlled_root)
print('PROJECT_ROOT', project_root)
print('RESULTS_ROOT', controlled_root)

In [ ]:
centroid_path = project_root / 'artifacts' / 'train_centroids.npz'
if not centroid_path.exists():
    subprocess.run([
        sys.executable, str(project_root / 'build_training_centroids.py'),
        '--embed-dir', str(project_root / 'VCTK_2F2M' / 'embeds'),
        '--output', str(centroid_path),
    ], check=True)
required = [
    project_root / 'VCTK_2F2M',
    project_root / 'checkpts/spk_encoder/enc.pt',
    project_root / 'real_last_cycle_train_dec_4speakers_original/vc_50_0823.pt',
    centroid_path,
    project_root / 'launch_layer4_kaggle.sh',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Missing migration assets: {missing}'
print('ASSET_CHECK_OK')

In [ ]:
command = ['bash', str(project_root / 'launch_layer4_kaggle.sh')]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
return_code = process.wait()
assert return_code == 0, f'Pipeline failed with exit code {return_code}'

In [ ]:
metrics_root = controlled_root / 'layer4_component_ablation' / 'metrics'
print('FINAL_ARTIFACT', working_root / 'layer4_component_ablation_outputs.tgz')
for path in sorted(metrics_root.glob('*')):
    print(path.name, path.stat().st_size)